# Computação Quântica na Prática — Nível Completo

> **Pré-requisitos:** os notebooks introdutório e intermediário.

Este é o nível mais avançado do workshop. O objetivo é uma visão abrangente e
prática do estado da arte, articulando os conceitos anteriores com temas de
pesquisa e aplicação atuais:

1. O **panorama** do hardware quântico e a era NISQ;
2. **Ruído e decoerência** — como os erros se acumulam;
3. **Mitigação de erros** — demonstração prática de correção de leitura;
4. **Teletransporte quântico** — implementado com circuito dinâmico e verificado;
5. **Algoritmos variacionais (VQE)** — otimização de circuitos parametrizados;
6. **Aplicações** — química quântica, otimização e o algoritmo de Shor;
7. **Projeto final** avaliado.

**Duração estimada:** um dia de trabalho.


In [ ]:
# %pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc seaborn
import math
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
import qutils
from grader import verificar, progresso
print("Ambiente carregado com sucesso.")

---
## 1. Panorama: a era NISQ

O estágio atual da tecnologia é designado **NISQ** (*Noisy Intermediate-Scale
Quantum*): processadores com dezenas a poucas centenas de qubits, sujeitos a ruído
e ainda sem correção de erros completa.

Diferentes tecnologias de qubit encontram-se em desenvolvimento:

| Tecnologia | Exemplos | Característica |
|---|---|---|
| Supercondutores | IBM, Google | Rápidos; requerem temperaturas próximas de 15 mK |
| Íons aprisionados | IonQ, Quantinuum | Alta fidelidade; operação mais lenta |
| Fotônica | Xanadu, PsiQuantum | Operam à temperatura ambiente |
| Átomos neutros | QuEra, Pasqal | Favoráveis ao aumento do número de qubits |

O objetivo de longo prazo é o **computador quântico tolerante a falhas**, dotado de
**correção de erros** — em que muitos qubits físicos codificam um qubit lógico
confiável. Esse patamar ainda não foi alcançado; por isso, no presente, recorre-se
largamente à **mitigação de erros**.

---
## 2. Ruído e decoerência

O ruído tem várias origens: **erros de porta** (cada operação é imperfeita),
**erros de leitura** (a medição ocasionalmente registra o valor errado) e
**decoerência** — a perda gradual da informação quântica devido à interação com o
ambiente, caracterizada pelos tempos $T_1$ (relaxação de energia) e $T_2$ (perda de
fase).

Uma consequência prática é que os erros **se acumulam com a profundidade** do
circuito: quanto mais operações, maior a degradação. A célula abaixo ilustra o
efeito aplicando a porta X um número **par** de vezes — o que, idealmente, equivale
à identidade e deveria sempre retornar `0`. No hardware (aqui, um simulador com
ruído de um chip real), a probabilidade de obter o resultado correto diminui à
medida que a profundidade aumenta.

> As barreiras impedem que o compilador cancele os pares `X;X`, preservando a
> profundidade real do circuito.

In [ ]:
profundidades, prob_correta = [], []
for pares in [1, 10, 30, 60, 100]:
    qc = QuantumCircuit(1)
    for _ in range(2 * pares):
        qc.x(0); qc.barrier()      # 2*pares portas X = identidade, no caso ideal
    qc.measure_all()
    contagens = qutils.simular_com_ruido(qc, shots=4000)
    total = sum(contagens.values())
    profundidades.append(2 * pares)
    prob_correta.append(contagens.get("0", 0) / total)
    print("%3d portas -> P(resultado correto = 0) = %.3f" % (2 * pares, prob_correta[-1]))

plt.figure(figsize=(7, 4))
plt.plot(profundidades, prob_correta, "o-")
plt.title("Acúmulo de ruído com a profundidade do circuito")
plt.xlabel("número de portas"); plt.ylabel("P(resultado correto)")
plt.ylim(0.9, 1.0); plt.grid(alpha=0.3); plt.show()

---
## 3. Mitigação de erros: correção de leitura na prática

Há duas famílias de resposta ao ruído:

- **Correção de erros (QEC):** emprega qubits adicionais para detectar e corrigir
  erros em tempo real, protegendo a informação. É o objetivo final, porém exige um
  número de qubits ainda indisponível na prática.
- **Mitigação de erros:** executa o circuito como está e aplica **pós-processamento
  estatístico** para estimar e reduzir o efeito do ruído nos resultados. Não corrige
  o hardware, mas melhora as estimativas com os recursos atuais.

A seguir, uma técnica concreta de mitigação: a **correção de erro de leitura**.
Primeiro calibramos o detector, preparando estados conhecidos (`0` e `1`) e medindo
com que frequência são lidos incorretamente — o que define uma **matriz de
confusão**. Em seguida, invertemos essa matriz para corrigir a distribuição medida
de um estado de interesse.

In [ ]:
def preparar_e_medir(prep, shots=8000):
    qc = QuantumCircuit(1)
    if prep == "1":
        qc.x(0)
    qc.measure_all()
    c = qutils.simular_com_ruido(qc, shots=shots); t = sum(c.values())
    return np.array([c.get("0", 0) / t, c.get("1", 0) / t])

# Calibração: como |0> e |1> são efetivamente lidos
coluna_0 = preparar_e_medir("0")
coluna_1 = preparar_e_medir("1")
A = np.column_stack([coluna_0, coluna_1])
print("Matriz de confusão (colunas = estado preparado):")
print(np.round(A, 3))

# Estado de interesse: |+>, cujo resultado ideal é 50/50
qc = QuantumCircuit(1); qc.h(0); qc.measure_all()
c = qutils.simular_com_ruido(qc, shots=8000); t = sum(c.values())
bruto = np.array([c.get("0", 0) / t, c.get("1", 0) / t])
corrigido = np.linalg.solve(A, bruto)

print("\nDistribuição medida (bruta):", np.round(bruto, 3))
print("Distribuição corrigida     :", np.round(corrigido, 3), "  (ideal: 0.5 / 0.5)")

### Exercício 1 — o que é mitigação de erros

- **(a)** Adicionar qubits extras para corrigir erros em tempo real.
- **(b)** Resfriar ainda mais o chip para eliminar fisicamente o ruído.
- **(c)** Usar pós-processamento estatístico para estimar e reduzir o efeito do ruído
  nos resultados.
- **(d)** Executar o circuito uma única vez para não acumular erros.

In [ ]:
resposta = ""   # "a", "b", "c" ou "d"
verificar("full-1", resposta)

---
## 4. Teletransporte quântico

O teletransporte quântico transfere o **estado** de um qubit para outro, sem
transportar matéria ou energia. Requer dois recursos: um par **emaranhado**
previamente compartilhado e a transmissão de **dois bits clássicos**. O estado
original é destruído no processo, em conformidade com o **teorema da não-clonagem**.

O protocolo, com três qubits (q0 = estado a transmitir; q1, q2 = par emaranhado):

1. Prepara-se o par emaranhado entre q1 e q2 (H e CNOT);
2. Interage-se q0 com q1 (CNOT e H) e medem-se q0 e q1;
3. Conforme os **dois bits** medidos, aplicam-se correções condicionais (X e/ou Z)
   sobre q2 — o que exige um **circuito dinâmico** (`if_test`).

Para verificar que a transferência ocorreu, prepara-se em q0 um estado conhecido com
`Ry(θ)` e, ao final, aplica-se `Ry(-θ)` a q2: se o estado foi corretamente
teletransportado, q2 retorna a `0` com certeza.

In [ ]:
theta = 1.0   # define o estado a ser teletransportado (arbitrário)

tp = QuantumCircuit(3, 3)
tp.ry(theta, 0)                       # estado desconhecido em q0
tp.barrier()
tp.h(1); tp.cx(1, 2)                   # par emaranhado q1-q2
tp.cx(0, 1); tp.h(0)                   # interação de q0 com q1
tp.measure(0, 0); tp.measure(1, 1)    # medição dos dois bits clássicos
with tp.if_test((1, 1)):              # correção condicional X
    tp.x(2)
with tp.if_test((0, 1)):              # correção condicional Z
    tp.z(2)
tp.barrier()
tp.ry(-theta, 2)                      # desfaz a preparação: se teleportou, q2 -> 0
tp.measure(2, 2)

contagens = qutils.simular(tp, shots=2000)
# o bit de q2 é o mais à esquerda na bitstring de 3 bits
q2_zero = sum(v for k, v in contagens.items() if k[0] == "0")
print("Fração de medições com q2 = 0:", round(q2_zero / sum(contagens.values()), 3))
print("Valor próximo de 1.0 confirma que o estado foi teletransportado para q2.")

### Exercício 3 — o que é transmitido

No teletransporte quântico, o que é reconstruído no qubit de destino?

- **(a)** A matéria física do qubit original.
- **(b)** O estado quântico do qubit original.
- **(c)** Uma cópia perfeita, preservando também o qubit original.
- **(d)** Energia, a uma velocidade superior à da luz.

In [ ]:
resposta = ""   # "a", "b", "c" ou "d"
verificar("full-3", resposta)

---
## 5. Amplificação de amplitude (Grover)

Retoma-se aqui a busca de Grover, com ênfase no princípio da **amplificação de
amplitude**: o oráculo marca a solução com uma inversão de fase, e o difusor
converte essa marcação em probabilidade observável. A célula abaixo marca o estado
`11` e confirma seu predomínio.

In [ ]:
grover = QuantumCircuit(2)
grover.h([0, 1])
grover.cz(0, 1)                                   # oráculo: marca |11>
grover.h([0, 1]); grover.x([0, 1]); grover.cz(0, 1); grover.x([0, 1]); grover.h([0, 1])
grover.measure_all()
print("Grover:", qutils.probabilidades(qutils.simular(grover, shots=3000)))
qutils.plotar(qutils.simular(grover, shots=3000), titulo="Amplificação de amplitude")

### Exercício 2 — o estado amplificado

O oráculo acima marcou o estado `11`. Após a amplificação, qual resultado domina a
medição? Responda como conjunto, por exemplo `{"11"}`.

In [ ]:
resposta = set()   # informe o conjunto
verificar("full-2", resposta)

### Exercício 5 — escalabilidade de Grover

Para $N = 8$ itens com uma solução, quantas iterações de Grover são ideais? Utilize
$(\pi/4)\sqrt{N}$ e arredonde.

In [ ]:
# Sugestão: math.pi, math.sqrt(8)
resposta = 0   # informe o inteiro
verificar("full-5", resposta)

---
## 6. Algoritmos variacionais (VQE)

Os **algoritmos variacionais** são a principal classe de métodos adequados aos
dispositivos NISQ. A ideia combina um circuito quântico **parametrizado** com um
**otimizador clássico**: mede-se uma grandeza (tipicamente a energia, dada por um
valor esperado), e o otimizador ajusta os parâmetros para minimizá-la. Essa é a base
do **VQE** (*Variational Quantum Eigensolver*), aplicado, por exemplo, ao cálculo da
energia do estado fundamental de moléculas.

A título de ilustração, considere o circuito de um qubit $\mathrm{Ry}(\theta)|0\rangle$
e a "energia" definida como o valor esperado $\langle Z \rangle$. Sabe-se que
$\langle Z \rangle = \cos\theta$. O gráfico abaixo mostra a energia em função de
$\theta$; o otimizador busca o mínimo.

In [ ]:
Z = Operator.from_label("Z")

def energia(theta):
    c = QuantumCircuit(1); c.ry(theta, 0)
    return np.real(Statevector.from_instruction(c).expectation_value(Z))

angulos = np.linspace(0, 2 * np.pi, 100)
energias = [energia(t) for t in angulos]

# "otimização" simples por varredura (um otimizador real faria isso de forma dirigida)
theta_otimo = angulos[int(np.argmin(energias))]
print("θ ótimo encontrado ≈ %.3f rad (esperado π ≈ %.3f)" % (theta_otimo, np.pi))
print("Energia mínima <Z> ≈ %.3f" % min(energias))

plt.figure(figsize=(7, 4))
plt.plot(angulos, energias, lw=2)
plt.axhline(-1, color="gray", ls="--", lw=1)
plt.scatter([theta_otimo], [min(energias)], color="red", zorder=5, label="mínimo")
plt.title(r"VQE (ilustração): energia $\langle Z \rangle = \cos\theta$")
plt.xlabel("θ (radianos)"); plt.ylabel(r"energia $\langle Z \rangle$")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Exercício 7 — a energia mínima

Para o estado $\mathrm{Ry}(\theta)|0\rangle$, a energia é $\langle Z \rangle =
\cos\theta$, que varia entre $-1$ e $1$. Qual é o **menor** valor de energia
alcançável? Responda com um número (por exemplo, `-1`).

In [ ]:
resposta = None   # informe o número
verificar("full-7", resposta)

---
## 7. Aplicações

- **Química e materiais quânticos** (VQE e sucessores): simulação de moléculas e
  reações. Como os sistemas de interesse já são quânticos, o computador quântico é a
  ferramenta natural. Impacto potencial em fármacos, baterias e catalisadores.
- **Otimização** (QAOA): rotas, cadeias de suprimento e portfólios — problemas com
  espaços de solução muito grandes.
- **Criptografia** (algoritmo de **Shor**): a fatoração eficiente de inteiros
  grandes comprometeria o RSA, o que motiva a transição para a **criptografia
  pós-quântica**.
- **Aprendizado de máquina quântico**: área emergente, ainda em avaliação quanto à
  vantagem efetiva.

### Exercício 4 — a aplicação mais natural

Qual das opções é considerada uma das aplicações mais promissoras e naturais de um
computador quântico?

- **(a)** Simular moléculas e materiais (química quântica).
- **(b)** Editar planilhas com maior rapidez.
- **(c)** Armazenar mais arquivos.
- **(d)** Renderizar vídeos em 4K.

In [ ]:
resposta = ""   # "a", "b", "c" ou "d"
verificar("full-4", resposta)

### Exercício 8 — Shor e a criptografia

O algoritmo de Shor fatora inteiros grandes de forma eficiente. Qual esquema de
criptografia amplamente utilizado tem sua segurança ameaçada por essa capacidade?

- **(a)** O RSA, cuja segurança depende da dificuldade de fatorar números grandes.
- **(b)** A senha do Wi-Fi doméstico.
- **(c)** A compressão de arquivos ZIP.
- **(d)** Nenhum — Shor não tem relação com criptografia.

In [ ]:
resposta = ""   # "a", "b", "c" ou "d"
verificar("full-8", resposta)

---
## 8. Projeto final: estado GHZ de 4 qubits

Como projeto de encerramento, generalize o padrão GHZ (visto com 2 e 3 qubits) para
**4 qubits**:

1. Hadamard no qubit 0;
2. cadeia de CNOTs: `cx(0,1)`, `cx(1,2)`, `cx(2,3)`.

O circuito deve produzir, na medição, apenas `0000` e `1111`.

In [ ]:
projeto = QuantumCircuit(4)

# Construa o estado GHZ de 4 qubits (1 Hadamard + 3 CNOTs encadeados):


projeto.measure_all()
projeto.draw("mpl")

In [ ]:
verificar("full-6", projeto)

In [ ]:
# Confirmação: apenas 0000 e 1111 devem ocorrer
print(qutils.probabilidades(qutils.simular(projeto, shots=3000)))
qutils.plotar(qutils.simular(projeto, shots=3000), titulo="Estado GHZ de 4 qubits")

---
## Conclusão

Este nível percorreu o ruído e sua mitigação, o teletransporte quântico
implementado com circuitos dinâmicos, os algoritmos variacionais e um panorama de
aplicações. Um próximo passo natural é executar esses circuitos em hardware real da
IBM (conforme o Módulo 7 do notebook introdutório) e aprofundar-se em bibliotecas
específicas — por exemplo, Qiskit Nature (química) e Qiskit Optimization.

Execute a célula abaixo para visualizar o progresso.

In [ ]:
progresso("full")